In [1]:
import os
import re
import numpy as np
import tifffile
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt
from natsort import natsorted
import pandas as pd
import glob

import matplotlib.patches as mpatches
import matplotlib.path as mpath 
from matplotlib.collections import PathCollection

In [5]:
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

# Function to calculate SSIM
def calculate_ssim(original_file, compressed_file):
    ssim_scores = []
    with tifffile.TiffFile(original_file) as tif_original:
        with tifffile.TiffFile(compressed_file) as tif_compressed:
            for original_frame, compressed_frame in zip(tif_original.series[0].pages, tif_compressed.series[0].pages):
                original_image = original_frame.asarray()
                compressed_image = compressed_frame.asarray()
                score = ssim(original_image, compressed_image, data_range=compressed_image.max() - compressed_image.min())
                ssim_scores.append(score)
    return np.median(ssim_scores) 

# Function to get the total size of files in a folder
def get_total_size_of_files(folder_path, file_extensions):
    total_size = 0
    for file in os.listdir(folder_path):
        if any(file.endswith(ext) for ext in file_extensions):
            total_size += os.path.getsize(os.path.join(folder_path, file))
    return total_size


def extract_k_lev(label):
    """
    Extracts the k and lev values from the label.
    Assumes the label format is like 'sparz_k5_rt35_lev0'.
    """
    k_match = re.search(r'_k(\d+)_', label)
    lev_match = re.search(r'_lev(\d+)', label)
    k_value = int(k_match.group(1)) if k_match else None
    lev_value = int(lev_match.group(1)) if lev_match else None
    return k_value, lev_value

In [9]:
### Astigmatims
# Change these paths according to your directory structure
main_folder = '/Users/laurabreimann/Desktop/astigmatism'
original_file = '/Users/laurabreimann/Desktop/astigmatism_raw/sequence-as-stack-MT0.N1.HD-AS-Exp.tif'

In [10]:

# Original file size
original_file_size = 20822904   # in bytes


# Color map for kernel sizes
k_colors = plt.cm.viridis(np.linspace(0, 1, 4))
lev_shades = np.linspace(0.5, 1, 4)
k_values = [5,7,9,11]

# Marker map for threshold values
threshold_markers = {35: 'o', 45: '^', 55: 's', 65: 'd'}

# Iterate through each subfolder and calculate median SSIM score and file size
results = {}
for folder in sorted(os.listdir(main_folder), key=natural_sort_key):
    if os.path.isdir(os.path.join(main_folder, folder)):
        uncompressed_folder = os.path.join(main_folder, folder, 'uncompressed')
        # Handling different file endings by searching for the common beginning
        for file_name in os.listdir(uncompressed_folder):
            if file_name.startswith('MT0_sequence-as-stack-MT0.N1'):
                compressed_file = os.path.join(uncompressed_folder, file_name)
                break
        median_ssim = calculate_ssim(original_file, compressed_file)
        total_file_size = get_total_size_of_files(os.path.join(main_folder, folder), ['.mp4', '.npz'])  
        file_size_percentage = (total_file_size / original_file_size) * 100
        k_value, lev_value = extract_k_lev(folder)
        threshold = int(re.search(r'rt(\d+)', folder).group(1))  # Extract threshold value

        # Choose marker based on threshold value
        marker = threshold_markers[threshold]

        color = k_colors[k_values.index(k_value)]

        # Choose shade based on lev value
        shade = lev_shades[lev_value] if lev_value is not None else 0.5

        if lev_value not in results:
            results[lev_value] = []
        results[lev_value].append({'median_ssim': median_ssim, 'file_size_percentage': file_size_percentage, 'color': color, 'shade': shade, 'marker': marker })



In [15]:
# Create facet plot
fig, axs = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
for i, (level, data_list) in enumerate(sorted(results.items())):
    ax = axs[i]
    ax.set_xlabel('Percentage of Original File Size')
    ax.set_title(f'Compression Level {level}')
    ax.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.2, zorder=0)
    for data in data_list:
        ax.scatter(data['file_size_percentage'], data['median_ssim'], color=data['color'],s=250, marker=data['marker'], alpha=0.9)
    ax.set_xlim(0, 50)

# Remove lines on top and right
for ax in axs:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Set common Y axis label
fig.text(0, 0.55, 'Median SSIM Score', va='center', rotation='vertical')

plt.tight_layout()
#plt.show()

# Save the plot as a PDF
output_file = '/Users/laurabreimann/Documents/Postdoc/Colabs/Compression/Figures/Figure_3_SSIM_filesize_240422.pdf'
plt.tight_layout()
plt.savefig(output_file, format='pdf')
plt.close()

In [3]:
### DNA PAINT
main_folder = '/Users/laurabreimann/Desktop/DNA-PAINT'
original_file = '/Users/laurabreimann/Desktop/DNA-PAINT_raw/RawFramesforTraining_P5-IS_20pM.tif'

In [10]:
# Original file size
original_file_size = 2644997930    # in bytes


# Color map for kernel sizes
k_colors = plt.cm.viridis(np.linspace(0, 1, 4))
lev_shades = np.linspace(0.5, 1, 4)
k_values = [11,13,15,17]

# Marker map for threshold values
threshold_markers = {5: 'o', 15: '^', 25: 's', 35: 'd'}

# Iterate through each subfolder and calculate median SSIM score and file size
results = {}
for folder in sorted(os.listdir(main_folder), key=natural_sort_key):
    if os.path.isdir(os.path.join(main_folder, folder)):
        uncompressed_folder = os.path.join(main_folder, folder, 'uncompressed')
        # Handling different file endings by searching for the common beginning
        for file_name in os.listdir(uncompressed_folder):
            if file_name.startswith('DNA-PAINT'):
                compressed_file = os.path.join(uncompressed_folder, file_name)
                break
        median_ssim = calculate_ssim(original_file, compressed_file)
        total_file_size = get_total_size_of_files(os.path.join(main_folder, folder), ['.mp4', '.npz'])  
        file_size_percentage = (total_file_size / original_file_size) * 100
        k_value, lev_value = extract_k_lev(folder)
        threshold = int(re.search(r'rt(\d+)', folder).group(1))  # Extract threshold value

        # Choose marker based on threshold value
        marker = threshold_markers[threshold]

        color = k_colors[k_values.index(k_value)]

        # Choose shade based on lev value
        shade = lev_shades[lev_value] if lev_value is not None else 0.5

        if lev_value not in results:
            results[lev_value] = []
        results[lev_value].append({'median_ssim': median_ssim, 'file_size_percentage': file_size_percentage, 'color': color, 'shade': shade, 'marker': marker })

In [12]:
# Create facet plot
fig, axs = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
for i, (level, data_list) in enumerate(sorted(results.items())):
    ax = axs[i]
    ax.set_xlabel('Percentage of Original File Size')
    ax.set_title(f'Compression Level {level}')
    ax.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.2, zorder=0)
    for data in data_list:
        ax.scatter(data['file_size_percentage'], data['median_ssim'], color=data['color'],s=250, marker=data['marker'], alpha=0.9)
    ax.set_xlim(0, 45)

# Remove lines on top and right
for ax in axs:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Set common Y axis label
fig.text(0, 0.55, 'Median SSIM Score', va='center', rotation='vertical')

plt.tight_layout()
#plt.show()

# Save the plot as a PDF
output_file = '/Users/laurabreimann/Documents/Postdoc/Colabs/Compression/Figures/Figure_3_SSIM_filesize_DNA-PAINT_240429.pdf'
plt.tight_layout()
plt.savefig(output_file, format='pdf')
plt.close()